# 2. Your own data: an XYZ geometry and a couplings file

Your molecule enters as an **XYZ file** (element and x, y, z in Å per atom) and your
measurements as a **CSV** (`spin_a, spin_b, D_obs_Hz, sigma_Hz`); see the
[file formats](../user/file-formats.md). This notebook uses the repository's tutorial files,
the synthetic biphenyl again, so that the answer is known (φ = 37°); replace the two paths with
your own files.

The steps are those of the app's Data tab: import the geometry, reconcile the couplings with it
and **confirm the mapping** of your labels to its protons, declare the torsion, choose the model,
fit.

In [1]:
from pathlib import Path

from dipolar_app.api import Studio

DATA = Path("../../datasets/biphenyl-synthetic")
XYZ_FILE = DATA / "biphenyl.xyz"  # your geometry
CSV_FILE = DATA / "tutorial-couplings.csv"  # your couplings

studio = Studio.local()
xyz = XYZ_FILE.read_text(encoding="utf-8")
studio.call("project.import_xyz", xyz=xyz, name=XYZ_FILE.stem, expected_science_revision=0)
spins = studio.call("project.describe")["spins"]
print(len(spins), "protons, named by atom number:", ", ".join(s["label"] for s in spins))

10 protons, named by atom number: H13, H14, H15, H16, H17, H18, H19, H20, H21, H22


## Reconcile, and confirm the mapping

Your file's labels (here `H2` … `H6'`) are not the imported protons' names (`H13` … `H22`).
`couplings.reconcile` proposes only exact matches and lists the rest; **you** decide the
mapping. Here the XYZ file's comment line names its atoms, which gives the mapping.

In [2]:
csv_text = CSV_FILE.read_text(encoding="utf-8")
proposal = studio.call("couplings.reconcile", csv=csv_text)
print("labels to map:", proposal["unmatched_labels"])

atom_names = xyz.splitlines()[1].split("atoms:")[1].split()  # from the comment line of this file
spin_of_atom = {s["atom"]: s["id"] for s in spins}
mapping = {label: spin_of_atom[atom_names.index(label)] for label in proposal["unmatched_labels"]}
diff = studio.call("couplings.reconcile", csv=csv_text, mapping=mapping)
assert diff["complete"], diff["unresolved_pairs"]
print(f"the import adds {len(diff['added'])} couplings")
imported = studio.call(
    "couplings.import", csv=csv_text, confirmed_mapping=mapping, expected_science_revision=diff["science_revision"]
)
print("imported", imported["n_couplings"], "couplings; ready to fit:", imported["ready_to_fit"])
print("  ", imported["not_ready_reason"])

labels to map: ['H2', "H2'", 'H3', "H3'", 'H4', "H4'", 'H5', "H5'", 'H6', "H6'"]
the import adds 14 couplings
imported 14 couplings; ready to fit: False
   imported geometry needs a declared torsion (a, i, j, b) to be fitted


## The torsion and the model

The fitted torsion is the dihedral of four atoms, counted from 0 in the XYZ file: here
C2–C1–C1′–C2′ (atoms 1, 0, 6, 7). For imported geometry the model is rigid, with the c2z (or
c1) order model in the input frame; the domain [−90°, 0°] holds the answer, −37° in this
dihedral's sign convention.

In [3]:
studio.call("molecule.set_torsion", torsion=[1, 0, 6, 7], expected_science_revision=studio.science_revision)
studio.call(
    "model.set",
    kind="rigid",
    order="c2z",
    frame="input",
    phi_domain=[{"value": -90, "unit": "deg"}, {"value": 0, "unit": "deg"}],
    expected_science_revision=studio.science_revision,
)
project = studio.call("project.describe")
print("ready to fit:", project["ready_to_fit"], project["not_ready_reason"] or "")

ready to fit: True 


In [4]:
run = studio.fit(intervals=True)
tau = run["theta"][0]["value"]
print(f"torsion = {tau:.3f} deg; chi2 = {run['chi2']:.2f} with nu = {run['nu']}; adequate: {run['adequate']}")
worst = max(run["couplings"], key=lambda c: abs(c["normalized_residual"]))
print(f"largest misfit: {worst['label']} at {worst['normalized_residual']:.2f} sigma")
assert abs(tau + 37.0) < 0.1  # the known answer, in this dihedral's convention
studio.close()

torsion = -36.988 deg; chi2 = 12.40 with nu = 10; adequate: True
largest misfit: H14–H19 at -1.94 sigma
